# LLM News Research Tool
## Notebook 7: Transfer and fresh-week tests

The pipeline is built and tuned on Tata Motors news only. This stage asks whether the same frozen pipeline works on news about a second company, Reliance Industries, with no change made for it. The pipeline runs as frozen: the Stage 1 rules, the default ranking (O1+S1) and the Stage 4 summary design (`summary_freeze.json`). The ranking comparison uses the retrieval-order baseline defined in Amendment 1. Reliance is a test set only: its labels and results do not change any setting.

Reliance Industries is a different kind of company from Tata Motors. It is a group with businesses in oil-to-chemicals, telecom and digital services, retail, consumer brands and new energy, so its news covers more sectors than the news about one carmaker.

The test covers one week of Reliance news (26 October to 1 November 2026) and one summary. A pass shows that the pipeline works on this week of news about this second company. It does not show that the pipeline works for every company or every week.

In [1]:
import csv, json, os, sys
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
from src import transfer_eval as te
from src.fresh_eval import file_fingerprint

pd.set_option("display.max_colwidth", 120)
RUN = te.RUN_DIR

def load(name):
    return json.loads((RUN / name).read_text(encoding="utf-8"))

## 1. The protocol

Everything the test measures is fixed in `transfer_protocol.json` and tagged `freeze-reliance-transfer` in Git before any article is fetched: the company scope, the time window, both searches, the sample rule, the definition of every measure, the pass levels and how the result is read. The fetch waits until 3 November, 06:00 UTC, because the free NewsAPI plan shows articles about one day late.

In [2]:
protocol = load(te.PROTOCOL_FILE)
print("Company:", protocol["company"])
print("Window:", " to ".join(protocol["window"]))
print("Fetch not before:", protocol["fetch_not_before"])
for name, query in protocol["searches"].items():
    print(f"Search {name}: {query}")
print("Search settings:", protocol["search_settings"])

ranking, summary = protocol["pass_levels"]["ranking"], protocol["pass_levels"]["summary"]
display(pd.Series({"configuration": ranking["configuration"], "primary k": ranking["primary_k"],
                   "reported k": ranking["reported_k"], "conditions": ", ".join(ranking["conditions"]),
                   "min model coverage": ranking["min_model_coverage"],
                   "max model seconds": ranking["max_model_seconds"]}, name="ranking").to_frame())
display(pd.Series({key: value for key, value in summary.items()}, name="summary").to_frame())
print("Reported without a level:", "; ".join(protocol["pass_levels"]["reported_without_level"]))

Company: Reliance Industries
Window: 2026-10-26T00:00:00Z to 2026-11-01T23:59:59Z
Fetch not before: 2026-11-03T06:00:00+00:00
Search A: "Reliance Industries"
Search B: ("Jio Platforms" OR "Reliance Jio" OR "Reliance Jio Infocomm" OR "Reliance Retail Ventures" OR "Reliance Retail" OR "Reliance Consumer Products" OR "Reliance New Energy")
Search settings: {'page_size': 30, 'sort_by': 'relevancy', 'exact_phrase': True, 'title_only': False}


,ranking
configuration,O1+S1
primary k,10
reported k,"[5, 15]"
conditions,"no_direct_event_lost, event_recall_not_lower, precision_not_lower"
min model coverage,0.9
max model seconds,180


,summary
design,the design frozen in Stage 4
k,5
produced,True
min_event_coverage,0.8
max_contradicted,0.05
no_central_contradiction,True


Reported without a level: the relevance check against the labels; unsure labels, on their own


**Company scope.** News about any of five group companies counts as news about Reliance Industries, including their subsidiaries and current products: Reliance Industries Limited, Jio Platforms Limited, Reliance Retail Ventures Limited, Reliance Consumer Products Limited and Reliance New Energy Limited. Jio Platforms lists on 28 October 2026, within the window, and stays under the control of Reliance Industries. Other companies with a similar name or from the same business group are separate businesses.

**Searches.** Search A asks for the exact name "Reliance Industries". Search B asks for the group companies and their main brands by name (Jio Platforms, Reliance Jio, Reliance Jio Infocomm, Reliance Retail Ventures, Reliance Retail, Reliance Consumer Products, Reliance New Energy), taken from the alias file. Both use the same settings: up to 30 articles each, sorted by relevance, exact phrases, searching the whole article.

**Sample.** Search A, then search B, merged in that order; each article is kept once by its tidy link; articles published outside the window or without a date are excluded and logged. The sample is what remains, even when it is small or unbalanced, with no top-up and no replacement. An article whose page cannot be fetched stays and is labelled on its short text.

**Pass levels.**

| Part | Condition | Level |
|---|---|---|
| Ranking: O1+S1 against the baseline at k = 10 (k = 5 and 15 reported) | no direct event lost | every direct event the baseline's top 10 covers is also covered by O1+S1's top 10, on the same articles and the same event names |
| | event recall not lower | O1+S1's event recall is not below the baseline's |
| | precision not lower | O1+S1's precision is not below the baseline's |
| Summary: the frozen Stage 4 design, k = 5 | produced | the one summary comes out |
| | event coverage | at least 80% |
| | contradicted items | at most 5% |
| | central contradiction | none |

The score reports each condition on its own. The scoring code, committed with Amendment 1 before any result exists, counts the ranking part as passed only when all three conditions hold, and the summary part only when all four of its levels hold.

| Model measure | Frozen level | Applies here? |
|---|---|---|
| Model coverage | at least 90% | No: O1+S1 does not use the model |
| Model time | at most 180 seconds | No: O1+S1 does not use the model |

The relevance check against the labels and the unsure labels are reported without a level.

**How the result is read.** Event recall is recall within the articles Stage 1 keeps, not within all news published about the company. The test judges one summary for factual support, event coverage and completion; it does not establish reliability across repeated searches. A failure is reported with its diagnosis; a fix is built and tested on Tata Motors development data only, then checked on a new Reliance week. A sample with few indirect or not-relevant articles is reported as it is; the searches are not changed.

## 2. Amendment 1: the baseline

The protocol compares O1+S1 with a baseline called O1. O1 orders the candidates, and S1 takes the first k of that order, so O1's top k and O1+S1's top k are the same list. The comparison passes on any data, so it gives no independent reference. Amendment 1 (`transfer_amendment.json`, tag `freeze-reliance-transfer-amend1`) replaces the baseline before any ranking, model or summary run on Reliance.

The new baseline is the retrieval order: search A's articles in the order NewsAPI returns them, then search B's, each article once, through the frozen Stage 1 rules (cleaning and duplicates) with no other sorting; the surviving articles keep their relative order, and the baseline takes the first k. The retrieval order is an independent reference because it preserves the order in which articles arrive before ranking. It gives a practical comparison for checking whether O1+S1 improves on the incoming order. It is not a neutral or universal baseline: search A comes before search B, and the order within each search comes from NewsAPI.

In [3]:
amendment = load(te.AMENDMENT_FILE)
protocol_text = (RUN / te.PROTOCOL_FILE).read_text(encoding="utf-8")
print("Baseline:", amendment["baseline"]["name"])
print("Definition:", amendment["baseline"]["definition"])
print("Configuration:", amendment["configuration"], "| conditions:", ", ".join(amendment["conditions"]))
print("Primary k:", amendment["primary_k"], "| reported k:", amendment["reported_k"])
print("Amends this protocol:", amendment["protocol_fingerprint"] == te._text_fingerprint(protocol_text))

Baseline: retrieval_order
Definition: search A's articles in the order returned, then search B's in the order returned; each article once; the frozen Stage 1 rules (cleaning and duplicates) applied with no other sorting; the surviving articles keep their relative order; the first k
Configuration: O1+S1 | conditions: no_direct_event_lost, event_recall_not_lower, precision_not_lower
Primary k: 10 | reported k: [5, 15]
Amends this protocol: True


The conditions, the k values and the definitions stay the same. Precision, event recall and direct-event recall of both lists are also reported at every k, without a level.

The amendment is not blind in the strictest sense: the labels and events already exist when it is written. It is written without reading them and before any ranking result exists, and its record holds the fingerprint of the protocol it amends.

## 3. Amendment 2: industry-context checks

The Stage 4 design adds an industry-context rule (prompt version 4, Stage 4 notebook section 6): an industry point needs an article that itself reports on the wider industry, the market or other companies, and a fact about the company alone is never stated as a trend for the sector. The protocol judges industry points only as items among all the others, so it cannot show whether this rule holds for a second company. Reliance Industries runs several businesses in different sectors, where a fact about one business is easy to state as a sector trend.

Amendment 2 (`transfer_amendment2.json`, tag `freeze-reliance-transfer-amend2`) adds three measures, judged with the rules of `SUMMARY_JUDGING_GUIDE_PART6.md`:

- **Industry evidence:** from the five selected articles alone, before the summary is read, does any article report a fact about the sector, the market or another company in its own right? The answer is yes or no, with a note naming the article and the fact, or the reason for no.
- **Industry points:** the industry points are classified as supported, partly supported or not supported. Whether any industry point is not supported is also reported.
- **Industry section:** the check that the section matches the evidence answer, with points only when there is evidence and an empty section only when there is none.

In [4]:
amendment2 = load(te.AMENDMENT2_FILE)
amendment_text = (RUN / te.AMENDMENT_FILE).read_text(encoding="utf-8")
for name, text in amendment2["added"].items():
    print(f"{name}: {text}")
print("Level:", amendment2["level"])
print("Unchanged:", amendment2["unchanged"])
print("Amends this protocol:", amendment2["protocol_fingerprint"] == te._text_fingerprint(protocol_text))
print("Follows this Amendment 1:", amendment2["amendment1_fingerprint"] == te._text_fingerprint(amendment_text))

industry_evidence: the answer to the industry-evidence question (yes or no), with a note naming the article and the fact, or the reason for no
industry_points: the industry points judged supported, partly supported and not supported, and the check that no industry point is not supported
industry_section: the check that the industry section matches the evidence answer: points only with evidence, and an empty section only without evidence
Level: all three are reported without a pass level and never decide whether the test passes
Unchanged: The ranking conditions, the summary pass levels (produced, event coverage, contradicted items, central contradiction) and every definition of the protocol and amendment 1 stay as they are.
Amends this protocol: True
Follows this Amendment 1: True


All three are reported without a pass level and never decide whether the test passes. The ranking conditions, the summary pass levels and every definition of the protocol and Amendment 1 stay as they are.

The amendment follows the Stage 4 fresh-week evaluation, which reports comparable industry-evidence and industry-support checks for Tata Motors. It is written before the Reliance run and before its summary exists. The labels and events are sealed, and the amendment does not read them.

## 4. The sample and the blind labels

Both searches return their full 30 articles. Merged by tidy link, 55 articles remain. No article falls outside the window or lacks a date, so the sample is all 55. For 49 articles the full page is fetched; for 6 the page cannot be fetched, and they are labelled on their short text. Stage 1 then applies its cleaning and duplicate rules during the run; the number of articles it keeps is part of the run record (section 6).

In [5]:
stages = json.loads((RUN / "state.json").read_text(encoding="utf-8"))["stages"]
display(pd.Series(stages["sample"]["counts"], name="articles").to_frame())
evidence = Counter(value.split(":")[0] for value in stages["sheet"]["evidence"].values())
print("Labelled on:", dict(evidence))

,articles
search_A,30
search_B,30
after_merge,55
outside_window,0
no_date,0
sample,55


Labelled on: {'full text': 49, 'short text': 6}


Every article is labelled blind with `LABELLING_GUIDE.md`, on a shuffled reading page that shows no search, filter or model output. Each relevant article then gets an event name. The labels and events are sealed in `transfer_seal.json` before any model runs on Reliance, and the score refuses to run if either file changes after the seal.

In [6]:
from src.evaluate import is_relevant
from src.fulltext import article_id
from src.summary_run import load_events

seal = load(te.SEAL_FILE)
print("Labels and events unchanged since the seal:",
      te.seal_changes(seal, file_fingerprint(RUN / te.LABEL_FILE), file_fingerprint(RUN / te.EVENTS_FILE)) == [])

with open(RUN / te.LABEL_FILE, encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle))
labels = {row["article_id"]: (row["label"] or "").strip().lower() for row in rows}
counts = Counter(labels.values())
display(pd.Series({"direct": counts["direct"], "indirect": counts["indirect"], "not relevant": counts["not"],
                   "total": len(labels)}, name="articles").to_frame())

events = {article_id(url): event for url, event in load_events(RUN / te.EVENTS_FILE).items()}
relevant = [key for key, label in labels.items() if is_relevant(label)]
by_event = defaultdict(set)
for key in relevant:
    by_event[events[key]].add(labels[key])
print("Relevant articles:", len(relevant), "| with an event name:", sum(1 for key in relevant if key in events))
print("Relevant events:", len(by_event), "| direct events:", sum(1 for found in by_event.values() if "direct" in found))
print("Unsure labels:", sum(1 for row in rows if (row.get("unsure") or "").strip().lower() == "yes"))

Labels and events unchanged since the seal: True


,articles
direct,47
indirect,4
not relevant,4
total,55


Relevant articles: 51 | with an event name: 51
Relevant events: 25 | direct events: 23
Unsure labels: 4


All 51 relevant articles (direct or indirect) have an event name. They describe 25 relevant events, 23 of which are direct events. Four labels are marked unsure; they are reported on their own in section 7.

The sample is mostly direct news: 47 of 55 articles. Under the protocol's balance rule, this is reported as it is, and the searches are not changed. With only four indirect and four not-relevant articles, the sample provides limited evidence about how well the ranking distinguishes these categories.

The summary is judged separately, against its own five articles (section 8).

## 5. Before the run: safeguards and budget

### 5.1 Safeguards in the code

The run (`python -m src.transfer_eval run`) refuses to start when:

- the run record, the attempt marker or the saved passages already exist;
- the protocol or either amendment differs from its tagged version in Git;
- the labels and events are not sealed;
- the Stage 4 freeze does not accept the current code and settings, or does not validate k = 5;
- Git shows an uncommitted change in a protected path: the code, the alias file, the Reliance folder, the judging guides or the freeze;
- the budget report is missing, says stop, or no longer matches the code, the inputs or the prompts of the run (section 5.2);
- the 24-hour confirmation is missing (section 5.3).

The run never reads the labels or events. Before any model request, it writes an attempt marker with its own attempt ID, the source commit and the confirmation. The marker is never removed, so a run that stops at any later point cannot be sent again, and a failure is not retried. The run then saves the exact passages the summary model receives to `data/fulltext/`, outside Git, with their fingerprints in the record. The saved passages and the run record carry the same attempt ID and commit, and the judging, the lock and the score refuse to go on if the three files do not match. The summary runs first, then a pause of 61 seconds, then the relevance check. Neither stage reads or writes an answer cache. The run prints counts and the summary status only, with no ranking result.

After the run, the judging sheets are built from the saved passages once their fingerprints are checked; nothing is rebuilt. The lock covers the sheets, the key, the judging code, the guides, the protocol, both amendments, the seal, the labels and events, the freeze, the run record, the attempt marker, the passages and the budget report. The run record stores the fingerprint of the budget report it starts from, and the judging, the lock and the score refuse to go on if the report changes. The score is made once, after the lock.

The safeguards are covered by tests in `tests/test_transfer_eval.py`.

In [7]:
print("Run record:", te.RUN_FILE, "| attempt marker:", te.ATTEMPT_FILE, "| budget report:", te.BUDGET_FILE)
print("Saved passages (outside Git): data/fulltext/" + te.PASSAGES_FILE)
print("Protected paths:")
for path in te.PROTECTED:
    print("  ", path)
print("Pause between the summary and the relevance check:", te.RELEVANCE_PAUSE, "seconds")
print("Confirmation flag:", te.CONFIRM_FLAG)

Run record: transfer_run.json | attempt marker: transfer_run_attempt.json | budget report: transfer_budget.json
Saved passages (outside Git): data/fulltext/transfer_passages_transfer_reliance_2026-10-26_11-01.json
Protected paths:
   src/
   requirements.txt
   data/company_aliases.json
   data/eval/transfer_reliance_2026-10-26_11-01/
   data/eval/LABELLING_GUIDE.md
   data/eval/SUMMARY_JUDGING_GUIDE.md
   data/eval/SUMMARY_JUDGING_GUIDE_PART4.md
   data/eval/SUMMARY_JUDGING_GUIDE_PART6.md
   data/eval/summary_freeze.json
   data/eval/summary_freeze_decision.md
Pause between the summary and the relevance check: 61 seconds
Confirmation flag: --no-groq-24h


### 5.2 Token budget

Groq's free plan allows 8,000 tokens a minute and 200,000 a day. Before the run, `python -m src.transfer_eval budget` builds and measures every prompt of the run without sending any request. Two rules must hold: every request, plus the largest extra amount Groq has been seen to count (551 tokens), must fit in one minute's limit; and the conservative total of the whole run must fit in the daily limit. The report is saved once, before the run, in `transfer_budget.json`, with the source commit and the fingerprints of the code, the inputs and every planned prompt; the run refuses if the report no longer matches.

In [8]:
budget = load(te.BUDGET_FILE)
plan, report = budget["plan"], budget["report"]
display(pd.Series({
    "candidates after Stage 1": plan["candidates"],
    "relevance batches before the split": plan["relevance_batches_before_split"],
    "relevance requests": report["requests"]["relevance_requests"],
    "largest relevance request (estimate)": max(report["requests"]["relevance_input_estimates"]),
    "per-request limit": report["per_request"]["limit"],
    "requests over the limit": len(report["per_request"]["over"]),
    "summary request (estimate)": report["requests"]["summary_input_estimate"],
    "conservative total": report["day"]["total"],
    "daily limit": report["day"]["limit"],
}, name="value").to_frame())
for name, text in report["assumptions"].items():
    if name != "token_bounds":
        print(f"- {name}: {text}")
print("Token bounds:", report["assumptions"]["token_bounds"])
print("Model requests:", budget["model_requests"], "| answer cache:", budget["answer_cache"])
print("Source commit:", budget["source_commit"])
print("Decision:", budget["decision"])

,value
candidates after Stage 1,53
relevance batches before the split,6
relevance requests,10
largest relevance request (estimate),6584
per-request limit,7449
requests over the limit,0
summary request (estimate),4875
conservative total,186976
daily limit,200000


- groq_extra: 551: the largest difference between Groq's requested tokens and its input count in its refusal messages; an observed value, not a proven maximum
- relevance_output: 1772 per request: the largest recorded relevance answer times 1.25 (the relevance requests have no output limit)
- relevance_tries: 2 per request (one try and one retry)
- relevance_split: a batch over the minute limit splits in two, as the frozen relevance check does; the split is planned with the conservative token bound
- summary: 2 full attempts and 4 repairs (the most repair requests recorded for one summary at k = 5), each at the summary's input estimate plus the full output limit of 4,096
- no_groq_24h: the whole daily limit is used only after 24 hours with no Groq use; the user confirms this when starting the run; the code cannot see the Groq usage
- estimate: a conservative estimate from recorded requests, not a guaranteed maximum
Token bounds: {'relevance': {'rate': 0.2388, 'base': 0.0, 'requests': 6

| Measure | Value |
|---|---|
| Relevance requests | 10: six batches of 10 articles, four of them too large for one request and split in two, as the frozen relevance check does |
| Largest relevance request (estimate) | 6,584 tokens, against a limit of 7,449 |
| Summary request (estimate) | 4,875 tokens |
| Conservative total for the run | 186,976 tokens, against a daily limit of 200,000 |

The total assumes every worst case at once: each relevance request answered twice, each answer as long as the longest recorded one plus a quarter, and the summary using two full attempts and four repairs, each with the full output limit of 4,096 tokens. The token counts come from a bound fitted to every recorded request of the same model. This is a conservative estimate from recorded requests, not a guaranteed maximum, and the margin under the daily limit is about 13,000 tokens.

The summary carries the pass levels; the relevance check is reported only. The summary therefore runs first, so that if the daily limit is reached, it reaches the part without a pass level.

### 5.3 The 24-hour condition

The budget can count the whole daily limit only when the account has no Groq use in the 24 hours before the run. The code cannot see the account's Groq usage. The run therefore needs the flag `--no-groq-24h`: the user's own statement that the condition holds. The attempt marker and the run record store it as a statement, not as a check. Without the flag, the run is refused.

## 6. Ranking result

Stage 1 keeps 53 of the 55 sample articles; the two it removes are both labelled not relevant. Both lists are scored against the sealed labels and events.

In [9]:
run = load(te.RUN_FILE)
score = load(te.SCORE_FILE)
kept = {c["key"] for c in run["candidates"]}
print(f"Candidates after Stage 1: {len(kept)}; removed:", {key: label for key, label in labels.items() if key not in kept})

ranking = score["ranking"]
rows = {}
for k in ("5", "10", "15"):
    for name in (ranking["baseline"], ranking["configuration"]):
        m = ranking["absolute"][name][k]
        rows[(int(k), name)] = {"precision": round(m["precision"], 2), "event recall": round(m["event_recall"], 2),
                                "direct-event recall": round(m["direct_event_recall"], 3),
                                "events covered": len(ranking["events_covered"][name][k])}
display(pd.DataFrame(rows).T.astype({"events covered": int}))
print(f"Conditions at k = {ranking['primary_k']}:", ranking["conditions"], "| passed:", ranking["passed"])
print("Direct events lost:", ranking["events"]["direct_lost"], "| gained:", ranking["events"]["direct_gained"])

Candidates after Stage 1: 53; removed: {'Ac0f925c479': 'not', 'A347e6c31a1': 'not'}


precision  event recall  direct-event recall  \
5  retrieval_order       0.80          0.12                0.130   
   O1+S1                 1.00          0.12                0.130   
10 retrieval_order       0.90          0.20                0.217   
   O1+S1                 1.00          0.20                0.217   
15 retrieval_order       0.87          0.24                0.261   
   O1+S1                 1.00          0.32                0.348   

                    events covered  
5  retrieval_order               3  
   O1+S1                         3  
10 retrieval_order               5  
   O1+S1                         5  
15 retrieval_order               6  
   O1+S1                         8

Conditions at k = 10: {'no_direct_event_lost': False, 'event_recall_not_lower': True, 'precision_not_lower': True} | passed: False
Direct events lost: ['jio_ipo_mobile_data_price_outlook'] | gained: ['ril_ncd_fundraising_announcement']


At k = 10, every article O1 + S1 selects is relevant (precision 1.00, against 0.90), and it covers as many events as the retrieval order: 5 of the 25 relevant events. But it loses one direct event the retrieval order covers, "jio_ipo_mobile_data_price_outlook", and gains another, "ril_ncd_fundraising_announcement". The protocol allows no lost direct event, so the ranking part fails: no direct event lost ✗, event recall not lower ✓, precision not lower ✓.

**Diagnosis.** The lost event has one article: "India's largest share sale could end cheap mobile internet for millions".

In [10]:
from src import ranking as rk

lost = [key for key, event in events.items() if event in ranking["events"]["direct_lost"] and key in kept]
order = [c["key"] for c in run["candidates"]]
for key in lost:
    title = next(c["title"] for c in run["candidates"] if c["key"] == key)
    print(f"{title} | label: {labels[key]} | place in the retrieval order: {order.index(key) + 1} of {len(order)}")
STAGE1_TEXTS_HERE = (ROOT / "data" / "fulltext").exists() and any((ROOT / "data" / "fulltext").glob("*.json"))
if STAGE1_TEXTS_HERE:
    from src.query import load_aliases
    candidates = te._run_candidates(load_aliases(te.ALIAS_FILE))
    o1 = [c["key"] for c in rk.order([dict(c, check=None) for c in candidates], "O1")]
    levels = {c["key"]: c["mention_level"] for c in candidates}
    for key in lost:
        print(f"Where the company is named: {levels[key]} (the body) | place in O1: {o1.index(key) + 1} of {len(o1)}")
    print("Where the company is named, O1's top 10:", Counter(levels[key] for key in run["selections"]["O1+S1"]["10"]))
else:
    print("Stage 1's full texts (data/fulltext/, outside Git) are not on this computer; the text below states "
          "where the article names the company and its place in O1.")

India's largest share sale could end cheap mobile internet for millions | label: direct | place in the retrieval order: 1 of 53


Where the company is named: content (the body) | place in O1: 47 of 53
Where the company is named, O1's top 10: Counter({'title': 10})


Its headline and description do not name Reliance or any group company; the company appears only in the body. The retrieval order puts it first. O1 sorts the candidates Stage 1 keeps by where the company is named (title, then description, then body; ties keep NewsAPI's order), so it falls to place 47 of 53, below every candidate that names the company in its headline. This is a property of O1, not of this one article: among the candidates, a relevant article whose headline does not name the company always ranks below every article whose headline does.

## 7. Relevance check and unsure labels

The relevance model answers all 53 articles in 549 seconds. The protocol reports this check without a pass level, and the ranking tested here, O1 + S1, does not use it.

In [11]:
from src.evaluate import is_relevant

check = score["relevance_check"]
answers = run["relevance"]["answers"]
said = {k for k, a in answers.items() if a.get("source") == "model" and is_relevant(a.get("label", ""))}
truly = {k for k in kept if is_relevant(labels[k])}
display(pd.Series({"articles answered": f"{check['answered']} of {check['articles']}",
                   "same label as the human label": f"{check['same_label']} of {check['answered']} ({check['agreement']:.1%})",
                   "called relevant by the model, and relevant": f"{len(said & truly)} of {len(said)} ({check['relevant_precision']:.1%})",
                   "relevant articles the model calls relevant": f"{len(said & truly)} of {len(truly)} ({check['relevant_recall']:.1%})",
                   "seconds": check["seconds"]}, name="value").to_frame())
titles = {c["key"]: c["title"] for c in run["candidates"]}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([{"title": titles[k], "label": labels[k], "model's answer": answers[k]["label"]}
                          for k in order if k in truly - said]))
print("Unsure labels:", [(u["label"], u["retrieval_order"], u["O1+S1"]) for u in score["unsure"]],
      "(label, selected by the retrieval order at k = 10, selected by O1 + S1 at k = 10)")

,value
articles answered,53 of 53
same label as the human label,46 of 53 (86.8%)
"called relevant by the model, and relevant",47 of 48 (97.9%)
relevant articles the model calls relevant,47 of 51 (92.2%)
seconds,549.2


,title,label,model's answer
0,Reliance Industries shares down 26% in 2026: Here’s how to trade RIL stocks on results day,direct,not
1,Jio’s Blockbuster I.P.O. Gives India’s Investors a Lift,direct,not
2,Jio Platforms Share Price & IPO Listing Live: Mukesh Ambani's Jio set for trading debut after record IPO,direct,not
3,Ambani's Jio surges 21% in Mumbai debut after record $3.2 billion India IPO,direct,not


Unsure labels: [('indirect', False, False), ('direct', False, False), ('indirect', False, False), ('indirect', False, False)] (label, selected by the retrieval order at k = 10, selected by O1 + S1 at k = 10)


The model misses 4 of the 51 relevant articles, and all four are labelled direct: three report Jio's stock market listing and one is the share-trading article ahead of the results. Used as a filter, the check would remove them. Stage 2 finds the same risk on Tata Motors with indirect industry news; here the missed articles are direct news about a group company. The check stays a ranking signal only, as Stage 3 decides.

Four labels are marked unsure: one direct and three indirect. Neither list selects any of them at k = 10, so they do not affect the ranking result.

## 8. Summary result

In [12]:
summary = score["summary"]
display(pd.Series({name: "pass" if ok else "FAIL" for name, ok in summary["checks"].items()}, name="result").to_frame())
items, covered = summary["values"]["items"], summary["values"]["events"]
print(f"Items: {items['total']} | supported {items['supported']}, partly supported {items['partly_supported']}, "
      f"not supported {items['not_supported']} | contradicted {items['contradicted']} of {items['total']}")
print(f"Events covered: {covered['covered']} of {covered['total']}")
result = run["summary_record"]["records"][0]["result"]
print(f"Model calls: {result['usage']['calls']} | repairs: {len(result['repairs'])} | points removed: {len(result['removed'])}")

,result
produced,pass
event_coverage,pass
contradicted,pass
no_central_contradiction,pass


Items: 11 | supported 8, partly supported 3, not supported 0 | contradicted 0 of 11
Events covered: 3 of 3
Model calls: 2 | repairs: 1 | points removed: 0


The summary passes every protocol level. Of its 11 items, 8 are supported and 3 partly supported; none is not supported. Two of the partly supported items give the Hamleys subsidiary's turnover for "FY 2025", where the article says calendar year 2025. The run uses two model calls, one of them a repair, and no point is removed.

**Industry context (Amendment 2, reported without a level).** The five articles contain industry evidence: one reports that global refining margins strengthened sharply amid supply disruptions in the Middle East and Russia.

In [13]:
industry = score["industry"]
print("Industry evidence:", industry["industry_evidence"], "| industry points:", industry["industry_points_produced"])
print({key: industry["industry_points"][key] for key in ("supported", "partly_supported", "not_supported")})
print("Reported checks:", industry["checks"])

Industry evidence: yes | industry points: 1
{'supported': 0, 'partly_supported': 1, 'not_supported': 0}
Reported checks: {'no_unsupported_industry_point': True, 'industry_section_matches_evidence': True}


The summary gives one industry point, judged partly supported. The evidence supports the stronger global margins, but the point adds that they contribute "to higher earnings for integrated refiners", a general claim the articles do not make; they report Reliance's own oil-to-chemicals gains. Both reported checks hold (no industry point is not supported, and the section matches the evidence), but the added claim is a mild form of the error the industry rule is written against: a company result stated as a trend for the sector.

## 9. Reading the result

The frozen summary design transfers to a second company. On Reliance Industries, a group with businesses in several sectors, the one summary covers every relevant event among its articles and contains no contradicted item. The test covers one week and one summary.

The default ranking fails one condition. Its top 10 holds only relevant articles and covers as many events as the retrieval order, but one direct event is lost because its only article does not name the company in the headline. The diagnosis points to a property of O1 that can occur for any company.

Under the protocol, nothing in the pipeline changes because of this result. A fix for O1's handling of headlines without the company name is built and tested on Tata Motors development data only, then checked on a new Reliance week. It belongs to the next development cycle, together with the event-coverage finding of Stage 4 and the threshold check of Stage 3. The industry point's added claim is investigated in that cycle as well, with no fix decided in advance.

## 10. Week 2: the final default on two fresh weeks

### 10.1 What is tested, and how

The second development cycle changes the default ranking to O1M5+S1 (Stage 3, section 8) and settles the summary design (Stage 4, section 10). Both choices rest on the three Tata Motors development days. Week 2 tests them once on one new week of Tata Motors news (31 October to 6 November 2026) and one new week of Reliance Industries news (2 to 8 November 2026).

Each week's protocol is fixed and tagged before its articles are fetched. It holds:

- the window;
- the same two searches as week 1;
- the sample rule;
- an overlap rule that excludes articles already in the company's earlier data;
- the measures and pass levels of Reliance week 1, with both of its amendments built in.

The labels and events are made blind, checked row by row, and sealed before any ranking or summary result exists. Nothing in the pipeline changes because of these weeks.

The ranking test uses no model. At k = 10, O1M5+S1 is compared with two lists:

- **the retrieval order** (primary): the same test as Reliance week 1, and it decides whether the test passes;
- **O1+S1** (second, reported only): the earlier default from Stage 3, on fresh data. It never changes the primary result.

### 10.2 Tata Motors week 2: the sample


In [14]:
WEEK2_TATA = te.EVAL_DIR / "week2_tata_2026-10-31_11-06"
week2_state = json.loads((WEEK2_TATA / "state.json").read_text(encoding="utf-8"))
with open(WEEK2_TATA / "labels.csv", encoding="utf-8") as handle:
    week2_label_rows = list(csv.DictReader(handle))
week2_labels = {r["article_id"]: r["label"] for r in week2_label_rows}
with open(WEEK2_TATA / "events.csv", encoding="utf-8") as handle:
    week2_event_rows = list(csv.DictReader(handle))
print("Searches and overlap:", week2_state["stages"]["overlap"]["counts"])
print("Text used for labelling:", dict(Counter(week2_state["stages"]["sheet"]["evidence"].values())))
print("Labels:", dict(Counter(week2_labels.values())))
print(f"Relevant articles: {len(week2_event_rows)}; events: {len({r['event'] for r in week2_event_rows})}")
print("Not relevant, of them car-auction listings (bringatrailer.com):",
      sum(r["label"] == "not" and "bringatrailer.com/listing" in r["url"] for r in week2_label_rows))
print("Candidates Stage 1 keeps:", json.loads((WEEK2_TATA / "week2_ranking.json").read_text(encoding="utf-8"))["candidates"])


Searches and overlap: {'search_A': 30, 'search_B': 30, 'after_merge': 60, 'earlier_link': 0, 'no_date': 0, 'outside_window': 0, 'to_review': 1, 'kept_before_review': 60}
Text used for labelling: {'full text': 56, 'short text: teaser': 1, 'short text: page not fetched': 3}
Labels: {'direct': 24, 'not': 23, 'indirect': 13}
Relevant articles: 37; events: 18
Not relevant, of them car-auction listings (bringatrailer.com): 16
Candidates Stage 1 keeps: 40


Both searches return their full 30 articles, and none is in the earlier data. One near-identical title goes to review: two auction listings of different cars, so both stay. For 56 of the 60 articles the full page is read; 4 are labelled on their short text. The labels are 24 direct, 13 indirect and 23 not relevant, 16 of them car-auction listings. The 37 relevant articles describe 18 events. Stage 1 keeps 40 candidates.

### 10.3 Tata Motors week 2: ranking result

The three orders come from the saved ranking record (`week2_ranking.json`): ordering again needs the article texts, which stay outside Git. Every list is measured again from those orders and the sealed labels and events, with the same measuring code, and checks the saved conditions against the result.


In [15]:
from src import ranking as rk
from src.evaluate import is_relevant
from src.fulltext import article_id

week2_ranking = json.loads((WEEK2_TATA / "week2_ranking.json").read_text(encoding="utf-8"))
week2_events = {article_id(r["url"]): r["event"].strip().lower() for r in week2_event_rows}
week2_keys = week2_ranking["orders"]["retrieval_order"]
week2_relevant = {k: is_relevant(week2_labels[k]) for k in week2_keys}
week2_lists = {"retrieval order": week2_keys[:10], "O1+S1": week2_ranking["orders"]["O1+S1"][:10],
               "O1M5+S1": week2_ranking["orders"]["O1M5+S1"][:10]}
week2_measured = {name: rk.measures(top, week2_keys, week2_relevant, week2_events, week2_labels, 10)
                  for name, top in week2_lists.items()}
display(pd.DataFrame([{"list (k = 10)": name, "article precision": round(m["precision"], 2),
                       "events covered": f"{len(m['events_covered'])} of {m['relevant_events']}",
                       "direct events covered": f"{len(m['direct_events_covered'])} of {m['direct_events']}"}
                      for name, m in week2_measured.items()]).set_index("list (k = 10)"))
for base, tested in (("retrieval order", "O1M5+S1"), ("O1+S1", "O1M5+S1"), ("retrieval order", "O1+S1")):
    change = rk.compare_events(week2_measured[base], week2_measured[tested])
    conditions = {"no direct event lost": not change["direct_lost"],
                  "event recall not lower": week2_measured[tested]["event_recall"] >= week2_measured[base]["event_recall"],
                  "article precision not lower": week2_measured[tested]["precision"] >= week2_measured[base]["precision"]}
    print(f"{tested} against {base}: {conditions}; gained {change['gained']}; lost {change['lost']}")
print("Saved record, primary:", week2_ranking["primary"]["conditions"], "| second:", week2_ranking["second"]["conditions"])
print("Events in O1M5+S1's top 10:", dict(Counter(week2_events.get(k, "(not relevant)") for k in week2_lists["O1M5+S1"])))
week2_untitled = week2_ranking["untitled_relevant"]
week2_lifted = [a for a in week2_untitled if a["in_top_10"]["O1M5+S1"]]
print(f"Relevant articles without the company in the title: {len(week2_untitled)}; in O1M5+S1's top 10: "
      f"{len(week2_lifted)}; in O1+S1's: {sum(a['in_top_10']['O1+S1'] for a in week2_untitled)}")
display(pd.DataFrame([{"title": a["title"], "event": week2_events[a["key"]],
                       "retrieval place": a["places"]["retrieval_order"], "O1+S1 place": a["places"]["O1+S1"],
                       "O1M5+S1 place": a["places"]["O1M5+S1"]} for a in week2_lifted if is_relevant(week2_labels[a["key"]])]))


,article precision,events covered,direct events covered
list (k = 10),,,
retrieval order,1.0,7 of 18,5 of 10
O1+S1,1.0,5 of 18,5 of 10
O1M5+S1,1.0,6 of 18,6 of 10


O1M5+S1 against retrieval order: {'no direct event lost': True, 'event recall not lower': False, 'article precision not lower': True}; gained ['tata_motors_share_price_surge']; lost ['dhanteras_festive_auto_deliveries', 'october_vahan_registrations']
O1M5+S1 against O1+S1: {'no direct event lost': True, 'event recall not lower': True, 'article precision not lower': True}; gained ['chandrasekaran_meloni_iveco_meeting']; lost []
O1+S1 against retrieval order: {'no direct event lost': False, 'event recall not lower': False, 'article precision not lower': True}; gained ['tata_motors_share_price_surge']; lost ['chandrasekaran_meloni_iveco_meeting', 'dhanteras_festive_auto_deliveries', 'october_vahan_registrations']
Saved record, primary: {'no_direct_event_lost': True, 'event_recall_not_lower': False, 'precision_not_lower': True} | second: {'no_direct_event_lost': True, 'event_recall_not_lower': True, 'precision_not_lower': True}
Events in O1M5+S1's top 10: {'cert_in_ira_app_flaws': 2, 'tata

,title,event,retrieval place,O1+S1 place,O1M5+S1 place
0,Tata-Iveco aims for 6 lakh annual vehicle sales in five years; India assembly plans underway,tata_iveco_turin_plan,2,25,2
1,Tata Sons chair Chandrasekaran meets Italy PM Meloni after Iveco takeover,chandrasekaran_meloni_iveco_meeting,6,12,5
2,TaMo looks at manufacturing Iveco in India,tata_iveco_turin_plan,9,13,6
3,Chandrasekaran allays fears of Iveco plant closures ahead of union talks,tata_iveco_turin_plan,11,28,8
4,TaMo-Iveco to create world's 4th largest CV group with ₹2.28 trn revenue,tata_iveco_turin_plan,12,14,9


**Primary: O1M5+S1 against the retrieval order, not passed.** It retains every direct event found by the retrieval order and adds one, the share-price rise (`tata_motors_share_price_surge`), so its direct-event recall is higher: 6 of 10 direct events against 5 of 10 (0.60 against 0.50). Its article precision, the share of selected articles that are relevant, stays at 1.00, the same as the retrieval order. Its overall event recall is lower: 6 of 18 events against 7 of 18. It loses two indirect market events, the Dhanteras deliveries and the October registration data, whose round-up articles do not name Tata Motors in the headline.

**Second, reported only: O1M5+S1 against O1+S1.** The same three conditions hold: no direct event lost, event recall not lower, and article precision not lower. O1M5 adds a direct event, the Iveco chairman's meeting with Italy's prime minister, and loses none. On fresh news, the change from O1 to O1M5 does what it is chosen for. This comparison does not change the primary result.

**A separate analysis: headlines without the company name.** This is not a pass condition. Of 25 relevant articles whose headline does not name Tata Motors, O1M5 brings 5 into the top 10 and O1 brings none. Four report the Turin press conference on the combined Tata Motors–Iveco plan, and one the meeting with Italy's prime minister.

**Diagnosis.** Four of O1M5+S1's top 10 articles report the same event, the Turin plan. S1 takes the first ten articles without grouping articles about the same news. On this week, that lets one large story occupy several places and pushes out the two market round-ups that appear in the retrieval order's top 10. The retrieval order happens to be more varied on this week. This points to a limitation in S1, not a failure of O1M5's recovery of untitled articles. Stage 3 tests S2, which groups similar articles, for exactly this case, but does not make it the default. This week gives a reason to test grouping again in a later cycle. Under the protocol, nothing changes because of this result.


### 10.4 Reliance Industries week 2: the sample

In [16]:
WEEK2_RELIANCE = te.EVAL_DIR / "week2_reliance_2026-11-02_11-08"
rel2_state = json.loads((WEEK2_RELIANCE / "state.json").read_text(encoding="utf-8"))
with open(WEEK2_RELIANCE / "labels.csv", encoding="utf-8") as handle:
    rel2_label_rows = list(csv.DictReader(handle))
rel2_labels = {r["article_id"]: r["label"] for r in rel2_label_rows}
with open(WEEK2_RELIANCE / "events.csv", encoding="utf-8") as handle:
    rel2_event_rows = list(csv.DictReader(handle))
with open(WEEK2_RELIANCE / "pairs.csv", encoding="utf-8") as handle:
    rel2_pairs = list(csv.DictReader(handle))
print("Searches and overlap:", rel2_state["stages"]["overlap"]["counts"])
print("Articles per day (UTC):", rel2_state["stages"]["fetch"]["day_counts"])
print("Text used for labelling:", dict(Counter(rel2_state["stages"]["sheet"]["evidence"].values())))
print("Labels:", dict(Counter(rel2_labels.values())))
print(f"Relevant articles: {len(rel2_event_rows)}; events: {len({r['event'] for r in rel2_event_rows})}; "
      f"largest event: {Counter(r['event'] for r in rel2_event_rows).most_common(1)[0]}")
print("Expectation-and-outcome pairs:", len(rel2_pairs))
print("Candidates Stage 1 keeps:", json.loads((WEEK2_RELIANCE / "week2_ranking.json").read_text(encoding="utf-8"))["candidates"])

Searches and overlap: {'search_A': 30, 'search_B': 30, 'after_merge': 55, 'earlier_link': 0, 'no_date': 0, 'outside_window': 0, 'to_review': 0, 'kept_before_review': 55}
Articles per day (UTC): {'A': {'2026-11-02': 21, '2026-11-03': 6, '2026-11-04': 0, '2026-11-05': 1, '2026-11-06': 2, '2026-11-07': 0, '2026-11-08': 0}, 'B': {'2026-11-02': 9, '2026-11-03': 6, '2026-11-04': 4, '2026-11-05': 2, '2026-11-06': 7, '2026-11-07': 2, '2026-11-08': 0}}
Text used for labelling: {'full text': 54, 'short text: teaser': 1}
Labels: {'direct': 28, 'not': 12, 'indirect': 15}
Relevant articles: 43; events: 18; largest event: ('reliance_q2_share_surge', 10)
Expectation-and-outcome pairs: 0
Candidates Stage 1 keeps: 49


Both searches return their full 30 articles. Five articles come up in both searches, so the merged sample holds 55. None is in the earlier data, none falls outside the window, and no pair of titles needs review. Search A finds 21 of its 30 articles on 2 November, the first trading day after the quarterly results. No article dated 8 November, a Sunday, appears among either search's results. The fetch runs 30 hours after the window ends, so articles from that day are eligible to be returned. Each search returns at most 30 articles, sorted by relevance. This shows only that no 8 November article ranks among the returned results, not that the day has no news.

For 54 of the 55 articles the full page is read; 1 is labelled on its short teaser. The labels are 28 direct, 15 indirect and 12 not relevant. The 43 relevant articles describe 18 events; the largest, the share rise on 2 November, has 10 articles. The week has no expectation-and-outcome pairs: the results come out before the window starts, so no article in it previews them. Stage 1 keeps 49 candidates.

### 10.5 Reliance Industries week 2: ranking result

The three lists are measured again in the same way as for Tata Motors.

In [17]:
from src import ranking as rk
from src.evaluate import is_relevant
from src.fulltext import article_id

rel2_ranking = json.loads((WEEK2_RELIANCE / "week2_ranking.json").read_text(encoding="utf-8"))
rel2_events = {article_id(r["url"]): r["event"].strip().lower() for r in rel2_event_rows}
rel2_keys = rel2_ranking["orders"]["retrieval_order"]
rel2_relevant = {k: is_relevant(rel2_labels[k]) for k in rel2_keys}
rel2_lists = {"retrieval order": rel2_keys[:10], "O1+S1": rel2_ranking["orders"]["O1+S1"][:10],
               "O1M5+S1": rel2_ranking["orders"]["O1M5+S1"][:10]}
rel2_measured = {name: rk.measures(top, rel2_keys, rel2_relevant, rel2_events, rel2_labels, 10)
                  for name, top in rel2_lists.items()}
display(pd.DataFrame([{"list (k = 10)": name, "article precision": round(m["precision"], 2),
                       "events covered": f"{len(m['events_covered'])} of {m['relevant_events']}",
                       "direct events covered": f"{len(m['direct_events_covered'])} of {m['direct_events']}"}
                      for name, m in rel2_measured.items()]).set_index("list (k = 10)"))
for base, tested in (("retrieval order", "O1M5+S1"), ("O1+S1", "O1M5+S1"), ("retrieval order", "O1+S1")):
    change = rk.compare_events(rel2_measured[base], rel2_measured[tested])
    conditions = {"no direct event lost": not change["direct_lost"],
                  "event recall not lower": rel2_measured[tested]["event_recall"] >= rel2_measured[base]["event_recall"],
                  "article precision not lower": rel2_measured[tested]["precision"] >= rel2_measured[base]["precision"]}
    print(f"{tested} against {base}: {conditions}; gained {change['gained']}; lost {change['lost']}")
print("Saved record, primary:", rel2_ranking["primary"]["conditions"], "| second:", rel2_ranking["second"]["conditions"])
print("Events in O1M5+S1's top 10:", dict(Counter(rel2_events.get(k, "(not relevant)") for k in rel2_lists["O1M5+S1"])))
rel2_untitled = rel2_ranking["untitled_relevant"]
rel2_lifted = [a for a in rel2_untitled if a["in_top_10"]["O1M5+S1"]]
print(f"Relevant articles without the company in the title: {len(rel2_untitled)}; in O1M5+S1's top 10: "
      f"{len(rel2_lifted)}; in O1+S1's: {sum(a['in_top_10']['O1+S1'] for a in rel2_untitled)}")
display(pd.DataFrame([{"title": a["title"], "event": rel2_events[a["key"]],
                       "retrieval place": a["places"]["retrieval_order"], "O1+S1 place": a["places"]["O1+S1"],
                       "O1M5+S1 place": a["places"]["O1M5+S1"]} for a in rel2_lifted if is_relevant(rel2_labels[a["key"]])]))
lost_event = "jio_comcast_charter_technology_talks"
print("Places of the lost event's articles:",
      [{name: rel2_ranking["orders"][name].index(k) + 1 for name in ("retrieval_order", "O1+S1", "O1M5+S1")}
       for k, event in rel2_events.items() if event == lost_event])

,article precision,events covered,direct events covered
list (k = 10),,,
retrieval order,0.8,4 of 18,4 of 12
O1+S1,0.8,3 of 18,3 of 12
O1M5+S1,0.8,4 of 18,3 of 12


O1M5+S1 against retrieval order: {'no direct event lost': False, 'event recall not lower': True, 'article precision not lower': True}; gained ['ambani_wealth_jump']; lost ['jio_comcast_charter_technology_talks']
O1M5+S1 against O1+S1: {'no direct event lost': True, 'event recall not lower': True, 'article precision not lower': True}; gained ['ambani_wealth_jump']; lost []
O1+S1 against retrieval order: {'no direct event lost': False, 'event recall not lower': False, 'article precision not lower': True}; gained []; lost ['jio_comcast_charter_technology_talks']
Saved record, primary: {'no_direct_event_lost': False, 'event_recall_not_lower': True, 'precision_not_lower': True} | second: {'no_direct_event_lost': True, 'event_recall_not_lower': True, 'precision_not_lower': True}
Events in O1M5+S1's top 10: {'(not relevant)': 2, 'reliance_q2_results': 2, 'ril_post_q2_share_outlook': 1, 'reliance_q2_share_surge': 4, 'ambani_wealth_jump': 1}
Relevant articles without the company in the title: 2

,title,event,retrieval place,O1+S1 place,O1M5+S1 place
0,Richest person in Asia: Mukesh Ambani vs Zhang Yinmin net worth — Reliance boss overtakes China's ByteDance TikTok head,ambani_wealth_jump,12,27,9
1,"Reliance shares surge 6%, eye best single-day gain since January 2024: What's driving the rally?",reliance_q2_share_surge,13,28,10


Places of the lost event's articles: [{'retrieval_order': 5, 'O1+S1': 24, 'O1M5+S1': 30}, {'retrieval_order': 29, 'O1+S1': 35, 'O1M5+S1': 37}, {'retrieval_order': 46, 'O1+S1': 22, 'O1M5+S1': 28}]


**Primary: O1M5+S1 against the retrieval order, not passed.** Both lists hold 8 relevant articles in their top 10 (article precision 0.80) and cover 4 of the 18 events. O1M5+S1 loses one direct event, the talks between Jio Platforms and the US cable companies Comcast and Charter, which the retrieval order has in fifth place. It gains one indirect event, the rise in Mukesh Ambani's wealth. Direct-event recall falls from 4 of 12 direct events to 3 of 12, so the condition "no direct event lost" fails.

**Second, reported only: O1M5+S1 against O1+S1.** All three conditions hold. O1M5 adds the wealth event and loses none. The earlier default, O1+S1, also loses the Comcast talks against the retrieval order, and covers only 3 events. This comparison does not change the primary result.

**A separate analysis: headlines without the company name.** Of 21 relevant articles whose headline does not name the company, O1M5 brings 2 into the top 10, at places 9 and 10: the wealth article and a report on the share rise. O1 brings none.

**Diagnosis.** The lost article's headline names the company only as "Jio" ("Ambani's Jio explores tech-for-equity talks"). The ranking counts "Jio Platforms", "Reliance Jio" and "RIL" as the company's names, but not "Jio" alone, because that word also matches Jio Financial Services, a separate company. So the article falls from place 5 in the retrieval order to place 24 under O1+S1 and place 30 under O1M5+S1. Both O1 orders share this loss; it is not caused by O1M5. Two articles that are not relevant hold places 1 and 4 in all three lists. Both name the company in the headline, but their text gives no information about it: a list of stocks with a technical signal, and a market report whose cleaned text never names the company. The cause differs from the Tata Motors week, where one large story fills several places. In both weeks, O1M5 beats O1+S1 and fails against the retrieval order. Under the protocol, nothing changes because of this result. Two separate experiments for a later cycle follow from the two weeks: grouping similar articles (S2), and counting a short name such as "Jio" when no other Jio company is named, with a check for false matches with Jio Financial Services.

### 10.6 The summaries of both weeks

Each week gets one summary run with the frozen design: direct, k = 5, prompt version 4, as confirmed by Stage 4 (section 10.3). The five articles are O1M5+S1's top 5 from the ranking test. Both runs produce a summary on the first call: 4,185 tokens for Tata Motors and 6,139 for Reliance Industries. The relevance check is not run, as the protocol states. The judging is blind and follows the week 1 guide; the sheets are locked before the key is opened. The summary test decides pass or fail; the industry result is reported separately, without a pass level.

In [18]:
week2_scores = {name: json.loads((folder / "week2_summary_score.json").read_text(encoding="utf-8"))
                for name, folder in (("Tata Motors", WEEK2_TATA), ("Reliance Industries", WEEK2_RELIANCE))}
summary_rows, industry_rows = {}, {}
for name, score in week2_scores.items():
    values, items = score["summary"]["values"], score["summary"]["values"]["items"]
    summary_rows[name] = {
        "summary produced": f"{values['produced']['count']} of {values['produced']['total']}",
        "key points and briefs supported": f"{items['supported']} of {items['total']}",
        "partly supported": items["partly_supported"],
        "contradicted items": items["contradicted"],
        "events covered": f"{values['events']['covered']} of {values['events']['total']} ({values['events']['coverage']:.2f})",
        "summary test": "passed" if score["summary"]["passed"] else "not passed",
    }
    industry = score["industry"]
    industry_rows[name] = {
        "industry evidence": industry["industry_evidence"],
        "industry points": industry["industry_points"]["total"],
        "supported": industry["industry_points"]["supported"],
        "partly supported": industry["industry_points"]["partly_supported"],
        "not supported": industry["industry_points"]["not_supported"],
        "level": industry["level"],
    }
print("Summary test (pass levels of week 1):")
display(pd.DataFrame(summary_rows))
print("Industry context (reported only):")
display(pd.DataFrame(industry_rows))
for name, score in week2_scores.items():
    print(name, "checks:", score["summary"]["checks"], "| prompt", score["prompt"], "| relevance check:",
          score["relevance_check"])

Summary test (pass levels of week 1):


,Tata Motors,Reliance Industries
summary produced,1 of 1,1 of 1
key points and briefs supported,9 of 10,11 of 13
partly supported,1,2
contradicted items,0,0
events covered,3 of 4 (0.75),2 of 2 (1.00)
summary test,not passed,passed


Industry context (reported only):


,Tata Motors,Reliance Industries
industry evidence,no,yes
industry points,0,2
supported,0,1
partly supported,0,1
not supported,0,0
level,reported without a pass level (amendment 2),reported without a pass level (amendment 2)


Tata Motors checks: {'produced': True, 'event_coverage': False, 'contradicted': True, 'no_central_contradiction': True} | prompt 4 | relevance check: not run
Reliance Industries checks: {'produced': True, 'event_coverage': True, 'contradicted': True, 'no_central_contradiction': True} | prompt 4 | relevance check: not run


**Tata Motors: not passed.** Event coverage is 0.75, below the pass level of 0.80. The summary cites all four events' articles, but the meeting between the Tata Sons chairman and Italy's prime minister appears in no point: the point drawn from that article states only the Iveco acquisition. Every other item is supported except one, which calls the Tata–Iveco business a "joint venture"; Tata owns 89.42% of Iveco. No article reports on the wider industry, and the industry section is correctly empty.

**Reliance Industries: passed.** Both events among the five articles are covered: the September-quarter results and the brokerages' outlook after them. No item is contradicted. Two of the five articles are not relevant to the company: the technical-signal list and the market report that never names it, both high in the ranking (section 10.5). The summary uses the market report for its two industry points. One is supported (the Nifty Metal index up more than 3%). The other adds a claim that the oil and gas rally "could benefit Reliance's oil-to-chemicals business", which the article does not make. The industry result is reported without a pass level, as in week 1.

**Across the two transfer tests,** both Reliance summaries pass and both Tata Motors summaries cover 3 of 4 events (0.75), below the 0.80 threshold. The two Tata misses differ: in the first fresh week, an outlook article (Stage 4, section 10.1); here, an article whose event becomes background to another point. These two weeks do not establish a lasting difference between the companies.

## 11. Limitations

### 11.1 Reliance Industries week 1

- **Small test.** One week and one summary: 53 candidates, 25 relevant events, 11 judged items and 3 events in the summary. The result does not establish reliability across weeks, or for companies beyond Tata Motors and Reliance Industries.
- **Unbalanced sample.** 47 of the 55 articles are labelled direct, so the test gives limited evidence on how well the ranking separates indirect and not-relevant news.
- **One judge.** The labels, the events and the summary judgements come from one judge.
- **Low recall by design.** The top 10 holds 10 of 53 candidates, so at most a small share of the week's 25 relevant events can appear in it; both lists cover 5. Event recall is measured within the articles Stage 1 keeps, not within all news about the company.
- **Free plans.** NewsAPI's Developer plan (a 24-hour delay, articles up to a month old, 100 requests a day) sets when the week can be fetched. On Groq's free plan (8,000 tokens a minute, 200,000 a day), the budget estimates room for one run of this size a day; Groq applies its limits to the whole account, and the 24-hour no-use condition rests on the user's statement, because the code cannot see the account's usage.
- **One sample.** The model writes with temperature 1.0, so another run can give a different summary and different relevance answers.

### 11.2 Week 2

- **One summary per week.** Each result is one run at temperature 1.0. Reliance's pass rests on two events and thirteen items.
- **The summary inherits the ranking.** The summary reads only the ranking's top 5, so two articles that are not relevant enter the Reliance summary; one of them produces an unsupported industry point.
- **No relevance check in week 2.** The protocol leaves the model relevance check out, so nothing filters those two articles before the summary.
- **Searches capped at 30 articles each,** sorted by relevance, so a missing day or story shows only what the searches return (section 10.4).
- **One judge** for labels, events and summary judging, as in week 1.